# 01. Data analysis of the benchmark logs

**Purpose.** Turn the processed benchmark measurements into validated statistics and publication figures, and make every step reproducible.

**How to run.** Open the repository in VS Code, select the `.venv` kernel, then use *Restart* and *Run All*. All paths are relative to the repository root, all random operations use the seed fixed below.

**Inputs.** `data/processed/measurements.csv` (individual measurements), `data/processed/metrics_summary.csv` (aggregated metrics produced by `src/process_data.py`) and, if present, `data/processed/measurements_load.csv` (load test).
**Outputs.** Tables in this notebook and figures in `reports/figures/`.

## 1. Setup and reproducibility

In [ ]:
import os, sys, random, platform
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Image

# Locate the repository root whether the notebook is started from notebooks/ or from the root
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

SEED = 42                      # fixed seed: every random operation below is reproducible
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.width", 140)
print("Repository root:", ROOT)
print("Python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__, "| seed", SEED)

## 2. Load the measurements and validate the schema

The table must contain at least the columns `phase`, `idx`, `latency_ms` and `status`. If one is missing the notebook stops with a clear message instead of silently producing wrong numbers.

In [ ]:
MEAS = ROOT / "data" / "processed" / "measurements.csv"
SUMM = ROOT / "data" / "processed" / "metrics_summary.csv"

df_all = pd.read_csv(MEAS)
required = {"phase", "idx", "latency_ms", "status"}
missing = required - set(df_all.columns)
assert not missing, f"Missing required columns: {sorted(missing)}. Found: {df_all.columns.tolist()}"

print("shape:", df_all.shape)
display(df_all.head())
print(df_all.dtypes.to_string())

## 3. Warm-up and errors (guardrail metric)

Warm-up records must not enter the final statistics. Errors are **not deleted**: their share is reported as a separate guardrail metric, so that dropping failed requests cannot bias the latency statistics (selection bias).

In [ ]:
print("records by phase:\n", df_all["phase"].value_counts(dropna=False).to_string())
print("\nrecords by status:\n", df_all["status"].value_counts(dropna=False).to_string())

MEASURE_PHASES = {"measure", "benchmark"}
bench = df_all[df_all["phase"].isin(MEASURE_PHASES)].copy()
is_ok = bench["status"].astype(str).str.lower().eq("ok")

n_req, n_err = len(bench), int((~is_ok).sum())
error_rate_pct = 100 * n_err / n_req
print(f"\nmeasured requests: {n_req} | errors: {n_err} | error rate: {error_rate_pct:.2f} % "
      f"| warm-up records excluded: {len(df_all) - len(bench)}")

bench_ok = bench[is_ok].copy()
bench_ok["latency_ms"] = pd.to_numeric(bench_ok["latency_ms"], errors="coerce")
assert bench_ok["latency_ms"].notna().all(), "non-numeric latency values found"
lat = bench_ok["latency_ms"]

## 4. Outliers: flagged, not removed

The 1.5 x IQR rule marks unusually slow or fast observations. They are only **flagged**: the cause (hardware noise, a real load spike, garbage collection, ...) is not known, so no observation is deleted and none is called an error.

In [ ]:
q1, q3 = lat.quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
flag = (lat < lo) | (lat > hi)
print(f"Q1={q1:.3f} ms | Q3={q3:.3f} ms | IQR={iqr:.3f} ms | fences=[{lo:.3f}, {hi:.3f}] ms | flagged: {int(flag.sum())} of {len(lat)}")

if "is_iqr_outlier" in bench_ok.columns:
    table_flag = bench_ok["is_iqr_outlier"].astype(str).str.lower().isin(["true", "1"])
    print("same observations as the flag column in the processed table:", bool((table_flag == flag).all()))

display(bench_ok.loc[flag, [c for c in ["idx", "t_utc", "latency_ms"] if c in bench_ok.columns]])

## 5. Aggregated statistics and bootstrap 95% confidence intervals

Mean, standard deviation, median (p50), p95 and p99 are recomputed here from the individual measurements and compared with `metrics_summary.csv`. Confidence intervals come from a percentile bootstrap (2000 resamples, seed fixed).

In [ ]:
x = lat.to_numpy()
point = {"mean": x.mean(), "sd": x.std(ddof=1), "p50": np.percentile(x, 50),
         "p95": np.percentile(x, 95), "p99": np.percentile(x, 99)}

B = 2000
rng = np.random.default_rng(SEED)
boot = x[rng.integers(0, len(x), size=(B, len(x)))]
boot_stats = {"mean": boot.mean(axis=1), "p50": np.percentile(boot, 50, axis=1),
              "p95": np.percentile(boot, 95, axis=1), "p99": np.percentile(boot, 99, axis=1)}
ci = {k: np.percentile(v, [2.5, 97.5]) for k, v in boot_stats.items()}

summ = pd.read_csv(SUMM)
row = summ[summ["phase"].isin(MEASURE_PHASES)].iloc[0] if "phase" in summ.columns else summ.iloc[0]
file_val = {"mean": row["latency_mean_ms"], "sd": row["latency_sd_ms"], "p50": row["latency_p50_ms"],
            "p95": row["latency_p95_ms"], "p99": row["latency_p99_ms"]}

table = pd.DataFrame({
    "recomputed here, ms": point,
    "metrics_summary.csv, ms": file_val,
})
table["difference, %"] = 100 * (table["recomputed here, ms"] / table["metrics_summary.csv, ms"] - 1)
display(table.round(4))

ci_table = pd.DataFrame({k: {"CI low, ms": v[0], "CI high, ms": v[1]} for k, v in ci.items()}).T
display(ci_table.round(3))
print("Point estimates agree with metrics_summary.csv within 1 %:", bool((table["difference, %"].abs() < 1).all()))

**Caveat on the intervals.** The bootstrap treats the measurements as independent. Consecutive runs of a program are often correlated (caches, CPU frequency, background load). The lag-1 autocorrelation below checks how serious this is; if it is clearly above zero, the intervals above are too narrow and should be read as a lower bound of the real uncertainty.

In [ ]:
ordered = bench_ok.sort_values("idx")["latency_ms"]
r1 = ordered.autocorr(lag=1)
print(f"lag-1 autocorrelation of latency in run order: {r1:.3f}")
if abs(r1) > 0.2:
    print("-> noticeable autocorrelation: the IID bootstrap intervals probably understate the uncertainty.")
else:
    print("-> weak autocorrelation: the IID assumption is not obviously violated, but it is still an assumption.")

## 6. Figure 2: distribution and outliers

Box plot of the individual measurements of the current implementation. Points flagged by the IQR rule keep their own marker. The y axis starts at 0.

In [ ]:
import make_figures as mf

OUT = ROOT / "reports" / "figures"
df_f, cols_f = mf.load_measurements(MEAS)
mf.figure2_distribution(df_f, cols_f, OUT, "Current implementation")
display(Image(filename=str(OUT / "fig2_distribution_boxplot.png")))

## 7. Figure 1: latency vs load (only if real load measurements exist)

The figure needs measurements at several request rates (RPS). Without a load column it is **not** drawn and nothing is substituted. The load test (`src/load_test.py`) sends requests on a fixed schedule to a local service that runs the analysis step; latency is measured from the intended send time.

In [ ]:
LOAD = ROOT / "data" / "processed" / "measurements_load.csv"
if LOAD.exists():
    df_l, cols_l = mf.load_measurements(LOAD)
    if mf.figure1_latency_vs_load(df_l, cols_l, OUT):
        display(Image(filename=str(OUT / "fig1_latency_vs_load.png")))
else:
    print("No data/processed/measurements_load.csv: Figure 1 was not produced (no load measurements available).")

In [ ]:
if LOAD.exists():
    raw = pd.read_csv(LOAD)
    raw["ok"] = raw["status"].astype(str).str.lower().eq("ok")
    rows = []
    for rps, g in raw.groupby("rps"):
        v = g.loc[g["ok"], "latency_ms"]
        rows.append({"rps": rps, "requests": len(g), "errors": int((~g["ok"]).sum()),
                     "error rate, %": 100 * (~g["ok"]).mean(),
                     "p50, ms": v.quantile(.50), "p95, ms": v.quantile(.95), "p99, ms": v.quantile(.99)})
    display(pd.DataFrame(rows).set_index("rps").round(2))

**How to read the load table.** At low rates the service keeps up and latency reflects the processing time. Once the request rate exceeds the capacity of the service, a queue builds up during the fixed duration of each level, so latency at those levels depends on how long the level lasted and is **not** a steady-state value. The rate at which the percentiles start to rise steeply marks the saturation point of this setup (client and server share one machine, so it is a relative, not an absolute, capacity).

## 8. Summary and limitations

In [ ]:
print(f"Current implementation, {len(lat)} successful measurements, error rate {error_rate_pct:.2f} %:")
print(f"  mean {point['mean']:.3f} ms (95% CI {ci['mean'][0]:.3f}-{ci['mean'][1]:.3f}), SD {point['sd']:.3f} ms")
print(f"  median {point['p50']:.3f} ms, p95 {point['p95']:.3f} ms, p99 {point['p99']:.3f} ms")
print(f"  IQR-flagged observations (kept): {int(flag.sum())}")

**Limitations.**
- Only one variant of the system was measured, so there is **no Baseline vs Proposed Method comparison** in this notebook.
- Bootstrap intervals assume independent measurements (see the autocorrelation check) and are therefore optimistic.
- The load test runs client and server on one machine and uses a synthetic coded table; the curve is relative.
- Outliers are flagged by a statistical rule; their cause has not been investigated.
- Results depend on the hardware and background load of the machine that produced the logs.